<a href="https://colab.research.google.com/github/Tanvi2307/The-Price-Is-Right/blob/main/agent2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q torch transformers accelerate bitsandbytes peft huggingface_hub chromadb sentence-transformers datasets litellm python-dotenv tqdm scikit-learn plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [6]:
!mkdir -p agents

In [7]:
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week8/agents/items.py -O agents/items.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week8/agents/agent.py -O agents/agent.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week8/agents/evaluator.py -O agents/evaluator.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week8/agents/deep_neural_network.py -O agents/deep_neural_network.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week8/agents/frontier_agent.py -O agents/frontier_agent.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week8/agents/neural_network_agent.py -O agents/neural_network_agent.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week8/agents/ensemble_agent.py -O agents/ensemble_agent.py

In [8]:
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
from google.colab import userdata
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed)
from peft import PeftModel

In [9]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [10]:
LITE_MODE = True
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  304kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [11]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)

In [12]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

(384,)


array([-5.68234101e-02, -6.70465603e-02,  4.41129729e-02,  5.98604744e-03,
       -2.28948668e-02, -2.95300744e-02,  5.56369089e-02,  3.42665575e-02,
       -1.08529389e-01, -3.81691046e-02, -7.43872523e-02, -1.03664294e-01,
        1.69148128e-02,  1.33925758e-03, -6.86191544e-02,  8.99353400e-02,
       -1.45186409e-02, -2.43885424e-02,  4.21854900e-03, -9.62992534e-02,
       -2.51799244e-02,  4.60675955e-02,  4.95349290e-03, -3.88679914e-02,
        1.07719435e-03,  6.82337582e-02, -1.13859745e-02, -5.83416298e-02,
       -1.03801340e-02, -1.74953323e-02, -1.86478551e-02,  4.07058466e-03,
        1.59437954e-02,  6.49722442e-02,  3.71175781e-02,  2.78224703e-02,
       -4.41945121e-02, -2.34373417e-02,  9.71035883e-02, -5.06139100e-02,
       -1.93864517e-02, -3.83471511e-02,  4.76066545e-02, -3.36107239e-02,
        5.08287437e-02,  3.57935429e-02,  2.91818241e-03, -1.06529169e-01,
        4.07211706e-02, -5.85415575e-04, -1.05607435e-01, -1.03584334e-01,
        3.71124297e-02, -

In [14]:
# Check if the collection exists; if not, create it

collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)): #"Go through train in groups of 1000 items."
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

  0%|          | 0/20 [00:00<?, ?it/s]

In [15]:
test[0]

<Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal = $219.0>

In [16]:
def vector(item):
  return encoder.encode(item.summary)

In [17]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5) #collection.query():Find products whose embeddings are most similar to this embedding."
    documents = results['documents'][0][:] #[:] just makes a copy of that list. It isn't doing any special filtering here.
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [18]:
find_similars(test[0])

(['Title: Digitech DOD Carcosa Fuzz Distortion Pedal  \nCategory: Electronics  \nBrand: DigiTech  \nDescription: An analog fuzz‑distortion pedal delivering classic to modern tones with standout treble and mid‑range clarity.  \nDetails: Features true‑bypass switching, multiple control knobs (BEFORE, OUTPUT, AFTER, HI‑CUT), a DEMHE‑HALI toggle for low‑mid boost, and a 9‑V power supply.',
  'Title: EarthQuaker Devices Plumes Small Signal Shredder – Purple Sparkle  \nCategory: Guitar Pedals  \nBrand: EarthQuaker Devices  \nDescription: An all‑analog overdrive pedal that delivers three distinct clipping voices and expansive headroom for crystal‑clear, three‑dimensional tone.  \nDetails: Features Flexi‑Switch relay‑based bypass, 9\u202fV power input, and a finely tuned tone control for sculpting low, mid, and high frequencies.',
  'Title: Dunlop Rotovibe Pedal  \nCategory: Audio Effects  \nBrand: Dunlop  \nDescription: A versatile chorus/vibrato pedal that simulates rotating speaker sounds w

In [19]:
def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [20]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
Title: Digitech DOD Carcosa Fuzz Distortion Pedal  
Category: Electronics  
Brand: DigiTech  
Description: An analog fuzz‑distortion pedal delivering classic to modern tones with standout treble and mid‑range clarity.  
Details: Features true‑bypass switching, multiple control knobs (BEFORE, OUTPUT, AFTER, HI‑CUT), a DEMHE‑HALI toggle for low‑mid boost, and a 9‑V power supply.
Price is $159.99

Potentially related product:
Title: EarthQuaker Devices Plumes Small Signal Shredder – Purple Sparkle  
Category: Guitar Pedals  
Brand: EarthQuaker Devices  
Description: An all‑analog overdrive pedal that delivers three distinct clipping voices and expansive headroom for crystal‑clear, three‑dimensional tone.  
Details: Features Flexi‑Switch relay‑based bypass, 9 V power input, and a finely tuned tone control for sculpting low, mid, and high frequencies.
Price is $99.00


In [21]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [22]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]["content"])

Estimate the price of this product. Respond with the price, no explanation

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
Title: Digitech DOD Carcosa Fuzz Distortion Pedal  
Category: Electronics  
Brand: DigiTech  
Description: An analog fuzz‑distortion pedal delivering classic to modern tones with standout treble and mid‑range clarity.  
Details: Features true‑bypass switching, multiple control knobs (BEFORE, OUTPUT, AFTER, HI‑CUT), a DEMHE‑HALI toggle for low‑

In [58]:
from google.colab import files

uploaded = files.upload()

Saving evaluator.py to evaluator.py


In [27]:
def gemini_rag(item):
    documents, prices = find_similars(item)
    response = completion(model = "gemini/gemini-flash-lite-latest", messages = messages_for(item, documents, prices))
    return response.choices[0].message.content

In [66]:
def gpt_oss_rag(item):
    documents, prices = find_similars(item)
    response = completion(model = "groq/openai/gpt-oss-20b", messages = messages_for(item, documents, prices), seed=42)
    return response.choices[0].message.content

In [24]:
test[0].price

219.0

In [25]:
from getpass import getpass
import os

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [28]:
gemini_rag(test[0])

'$229.00'

In [67]:
gpt_oss_rag(test[0])

'$249.99'

In [63]:
from evaluator import evaluate

In [64]:
evaluate(gemini_rag, test, size = 15, workers = 1)

  0%|          | 0/15 [00:00<?, ?it/s]

$10 $14 $5 $10 $10 $0 $64 $50 $6 $80 $18 $30 $5 $4 $44 



===== EVALUATION RESULTS =====
Average Error (MAE): $23.37
MSE: 1,118.86
R²: 96.50%


In [55]:
import re

def get_price(reply):

    reply = reply.replace("$", "")
    reply = reply.replace(",", "")

    match = re.search(
        r"[-+]?\d*\.\d+|\d+",
        reply
    )

    return float(match.group()) if match else 0

In [30]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
LORA_MODEL = "TanviAgarwal/price-2026-09-19_05.58.52-lite"
REVISION = "8a6c78d51f9c7ebf3b34be70f463644a358b4801"

In [31]:
#load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [32]:
#Load the base model in 4-bit:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [33]:
#Now attach the LoRA adapter
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL,
    revision=REVISION
)

adapter_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 73.4MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

In [34]:
def price(description: str) -> float:

    PREFIX = "Price is $"
    QUESTION = "What does this cost to the nearest dollar?"
    prompt = f"{QUESTION}\n\n{description}\n\n{PREFIX}"

    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = fine_tuned_model.generate(inputs, max_new_tokens=5)
    result = tokenizer.decode(outputs[0])

    contents = result.split("Price is $")[1] #Python splits the string wherever "Price is $" appears.So we get something like:["Question...\n", "599"]
    contents = contents.replace(",", "")

    match = re.search( #re.search() searches a string using a regular expression (regex).
        r"[-+]?\d*\.\d+|\d+",
        contents
    )
    # match is not directly the string "599".It is a Match object.To get the actual matched text, we use: match.group()
    return float(match.group()) if match else 0

In [35]:
def specialist(item):
    return price(item.summary)

In [4]:
from google.colab import files

uploaded = files.upload()

Saving deep_neural_network.pth to deep_neural_network.pth


In [36]:
from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()

runner.setup()

runner.load("deep_neural_network.pth")

In [37]:
def deep_neural_network(item):
    return runner.inference(item.summary)

In [69]:
def ensemble(item):

    price1 = get_price(gemini_rag(item))

    price2 = specialist(item)

    price3 = deep_neural_network(item)

    return (
        price1 * 0.8
        + price2 * 0.1
        + price3 * 0.1
    )

In [70]:
print(ensemble(test[0]))

238.89866809082034


In [71]:
evaluate(ensemble, test, size = 15)

  0%|          | 0/15 [00:00<?, ?it/s]

$2 $23 $9 $1 $11 $154 $65 $54 $6 $29 $91 $37 $7 $11 $31 



===== EVALUATION RESULTS =====
Average Error (MAE): $35.35
MSE: 2,883.43
R²: 90.98%
